<a href="https://colab.research.google.com/github/AM0ya/Clase_Programacion/blob/main/Sesion10_Data_Profiling_Entregable_MATRICULA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Alfredo Moya Orozco

**Matrícula:** 269712

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [8]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)


Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [19]:
# Tu código aquí
df_marketing_renombrado = df_marketing.copy()
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()
df_marketing_renombrado = df_marketing_renombrado.rename(columns={'year_birth': 'year_birth', 'kidhome': 'kid_home', 'mntwines': 'mnt_wines'})
print("Copia Data frame renombrado: \n\n")

df_marketing_renombrado.columns



Copia Data frame renombrado: 




Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kid_home',
       'teenhome', 'dt_customer', 'recency', 'mnt_wines', 'mntfruits',
       'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts',
       'mntgoldprods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [26]:
# Tu código aquí

pd.to_datetime(df_netflix['date_added'], format='mixed')
df_netflix.dtypes
df_netflix.isnull().sum()


,0
show_id,0
type,0
title,0
director,2389
cast,718
country,507
date_added,10
release_year,0
rating,7
duration,0


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [37]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [45]:
# Tu código aquí
print('Filas con duplicados inyectados:',df_marketing_dup.shape,"\n")
print("Suma de duplicados:",df_marketing_dup.duplicated().sum(),"\n")
print("Suma de duplicados de la columna ID:",df_marketing_dup.duplicated(subset='id').sum(),"\n")
df_marketing_dup = df_marketing_dup.drop_duplicates()

print('Filas con duplicados eliminados:',df_marketing_dup.shape)



Filas con duplicados inyectados: (2240, 29) 

Suma de duplicados: 0 

Suma de duplicados de la columna ID: 0 

Filas con duplicados eliminados: (2240, 29)


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [74]:
# Tu código aquí
print("Valores faltantes por columna:\n",df_netflix.isnull().sum(),"\n")

#se crea una copia filtrada de las filas con valores faltantes
filas_valores_faltantes = df_netflix[df_netflix.isnull().any(axis=1)]

#se imprime solo un overview de las filas con valores faltantes por mera representacion
#grafica ya que son una cantidad de datos considerablemente grandes.
print("Filas con al menos un valor faltante:\n")
display(filas_valores_faltantes.head(3))

#se imprime el numero total de filas faltantes
print(f"\nTotal de filas con al menos un valor faltante: {filas_valores_faltantes.isnull().any(axis=1).sum()}")

Valores faltantes por columna:
 show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64 

Filas con al menos un valor faltante:



,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,14-Aug-20,2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
11,s12,TV Show,1983,NaN,"Robert Więckiewicz, Maciej Musiał, Michalina O...","Poland, United States",30-Nov-18,2018,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Dramas","In this dark alt-history thriller, a naïve law..."
12,s13,TV Show,1994,Diego Enrique Osorno,NaN,Mexico,17-May-19,2019,TV-MA,1 Season,"Crime TV Shows, Docuseries, International TV S...",Archival video and new interviews examine Mexi...



Total de filas con al menos un valor faltante: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [78]:
# Tu código aquí
total_filas = len(filas_valores_faltantes)
completitud = (1 - filas_valores_faltantes.isnull().sum() / total_filas) * 100

print("RESPUESTA: La columna con el nivel de completitud mas bajo es la de directo con un 19.8%\n\n")
completitud


RESPUESTA: La columna con el nivel de completitud mas bajo es la de directo con un 19.8%




,0
show_id,100.000000
type,100.000000
title,100.000000
director,19.805304
cast,75.897952
country,82.980866
date_added,99.664317
release_year,100.000000
rating,99.765022
duration,100.000000


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

**RESPUESTA:** Para estas tres categorias encontradas que parecieran ser un error, se tendria que tener en cuenta que representan un sentimiento o estatus real de algunas personas, las cuales buscan representar su estado civil. Si bien no son un estandar en un formulario de datos, si representan un estatus unico. Por lo que "Alone" podria representar simplemente soltero, "Absurd" podria representar un estatus "Together", y "YOLO" representaria una exprecion moderna a estar en una relacion abierta ("Together") o que simplemente es soltero ("Single").

El tomar la desicion de eliminarlos no es opcion. Y dejarlos tal cual tampoco ayudaria a nuestro futuro modelo predictivo.
Buscaria investigar mas en el contexto social y **reclasificar** en donde encajan mejor esas categorias anormales

In [80]:
# Tu código aquí
df_marketing['marital_status'].value_counts()


,count
marital_status,
Married,864
Together,580
Single,480
Divorced,232
Widow,77
Alone,3
Absurd,2
YOLO,2


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [85]:
# Tu código aquí
porcentaje_consistente = df_netflix['show_id'].str.match(r'^s\d+$').mean() * 100

print(f'Porcentaje de IDs consistentes: {porcentaje_consistente:.2f}%')




Porcentaje de IDs consistentes: 100.00%


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

**RESPUESTA:** Para este caso consideraria un error de captura ya que en la columna de "dt_customer" presenta fechas muy actuales. Por lo que seria imposible que un cliente tuviera entradas de datos en el 2014 con una fecha de nacimiento tan antigua.

In [88]:
# Tu código aquí
print(df_marketing[['year_birth']].describe())
year_antiguos = df_marketing[df_marketing['year_birth'] < 1900]
year_antiguos


        year_birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000


,id,year_birth,education,marital_status,income,kidhome,teenhome,dt_customer,recency,mntwines,...,numwebvisitsmonth,acceptedcmp3,acceptedcmp4,acceptedcmp5,acceptedcmp1,acceptedcmp2,complain,z_costcontact,z_revenue,response
239,11004,1893,2n Cycle,Single,60182.0,0,1,2014-05-17,23,8,...,4,0,0,0,0,0,0,3,11,0
339,1150,1899,PhD,Together,83532.0,0,0,2013-09-26,36,755,...,1,0,0,1,0,0,0,3,11,0


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [116]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['income'] = df_practica['income'].astype(object)
df_practica.loc[5, 'income'] = 'sesenta mil'

df_practica[['id', 'marital_status', 'income']]

,id,marital_status,income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [117]:
# Paso 1 — ajuste de tipos
print("Tipo de datos antes de conversion:\n")
display(df_practica.dtypes[['id', 'marital_status', 'income']])
df_practica['income'] = pd.to_numeric(df_practica['income'], errors='coerce')
print("\nTipo de datos despues de conversion:\n")
df_practica.dtypes[['id', 'marital_status', 'income']]


# """nota:si se vuelve a ejecutar la imprecion de antes de la conversion cambiara
#  el dato a float ya que se queda guardado en la memoria. para no tener este el problema, hay que
# reestablecer los valores de la variable ejecutando la celda anterior"""



Tipo de datos antes de conversion:



,0
id,int64
marital_status,object
income,object



Tipo de datos despues de conversion:



,0
id,int64
marital_status,object
income,float64


In [118]:
# Paso 2 — duplicados
print("Total de valores duplicados:",df_practica.duplicated().sum(),"\n")
df_practica = df_practica.drop_duplicates()
print("Total despues de eliminacion de datos duplicados:",df_practica.duplicated().sum())


Total de valores duplicados: 1 

Total despues de eliminacion de datos duplicados: 0


In [133]:
# Paso 3 — valores faltantes
print("Valores faltantes por columna:\n",df_practica[['id', 'marital_status', 'income']].isnull().sum(),"\n")




Valores faltantes por columna:
 id                0
marital_status    0
income            1
dtype: int64 



In [134]:
# Paso 4 — exploración categórica
df_practica['marital_status'].unique()

# RESPUESTA:  Para esta columna no se requiere normalizacion de datos ya que son valores normales para un formulario de este tipo.

array(['Together', 'Single', 'Married', 'Divorced'], dtype=object)

**Tu reporte de profiling:**

*(Escribe aquí tu resumen de 3-4 líneas)*

El reporte de profiling para esta practica nos da mayor claridad sobre la informacion a tratar. Al tratarse de un sample de 15 filas, solo se encontro una evento con valor faltante, resultado de la conversion de tipos del tipo de dato agregado con tipo incorrecto. Al ser un solo dato se consideraria imputarlo y remplazarlo por el valor real.